### Agents

In [5]:
!pip install langchain_groq

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

In [6]:
PRODUCTS = {
    "wireless headphones":{"price":79.99,"rating":4.6,
     "description":"Over-ear Bluetooth,30-hr battery , active noise cancellation"},
     "smart watch": {"price":199.99,"rating" : 4.3,
      "description":"Tracks heart rate and sleep , 5-day battery,water-resistant"},
     "mechanical keyboard": {"price" : 129.00,"rating":4.8,
    "description" : "Tenkeysless , cherry MX Brown switches,per- key RGB."},
    "laptop stand":{"price":34.99,"rating":4.5,
                    "description":"Adjustable aluminium ,fits  11-17 inch laptops , folds for"}
}
@tool
def get_product(name:str) -> str:
  """ Look up a product by name and return its price , rating
  and description. """
  p = PRODUCTS.get(name.lower())
  if not p:
    return f"Product not found . Available : {', '.join(PRODUCTS)}"
  return str(p)

In [ ]:
llm = ChatGroq(model = "openai/gpt-oss-120b",api_key = "your - Api - Key")
pmt= "You are a helpful product assistant for an online tech store."
agent = create_agent(
    llm,
    tools = [get_product],
    system_prompt=pmt
)


In [8]:
def ask(question : str):
  result = agent.invoke({"messages" : [{"role" : "user",
                                        "content": question}]})

  print(result["messages"][-1].content)

In [9]:
ask("what is the price of wireless headphone.")

The wireless headphones are priced at **$79.99**.


In [10]:
ask("tell me rating")

Sure! Could you let me know which product you’d like the rating for? Just give me the name of the item, and I’ll pull up its rating (and any other details you need).


In [11]:
REVIEWS = {
    "wireless headphones": {"review": 1262,"rating": 4.6},
    "smart watch" : {"reviews" : 340, "rating" : 3.9},
    "mechanical keyboard" : {"reviews" : 57 , "rating" : 4.8},
    "laptop stand" : {"reviews" : 12 , "rating" : 4.5}
}
@tool
def get_review(name:str) -> str:
  """look up a product review by a product name.
  Return the product name , number of reviews and rating """
  r= REVIEWS.get(name.lower())
  if not r :
    return " Review not available for this product "
  return str(r)

In [12]:
pmt= "You are a helpful product assistant for an online tech store."
agent2 = create_agent(
    llm,
    tools = [get_product,get_review],
    system_prompt=pmt
)
def ask2(question : str):
  result = agent2.invoke({"messages" : [{"role" : "user",
                                        "content": question}]})

  print(result["messages"][-1].content)

In [13]:
ask2("how do people like smart watch")

People seem to have a generally positive but mixed opinion about the **smart watch**:

- **Average rating:** **3.9 / 5**  
- **Number of reviews:** **340**

A 3.9 rating suggests that most owners are satisfied with the core features—such as fitness tracking, notifications, and battery life—though there are enough critiques (e.g., UI quirks, occasional connectivity issues) to keep the score just under 4. If you’re weighing whether to buy one, it’s a solid mid‑range choice, especially if the specific features you need (like heart‑rate monitoring or app support) are covered.


In [14]:
ask2("what is the price and reviews of smart watch")

**Smart Watch**

- **Price:** $199.99  
- **Customer Reviews:** 340 reviews  
- **Overall Rating:** The product page lists a rating of **4.3 ★**, while the aggregated customer reviews show a rating of **3.9 ★**.

If you’d like more details about features, warranty, or need help placing an order, just let me know!


In [15]:
ask2("what is the price of wireless headphones.")

The wireless headphones are priced at **$79.99**.


In [16]:
ask2("what are the reviews on this product")

I’m happy to help with the reviews, but I need to know which product you’re referring to. Could you please tell me the name of the product you’d like to see reviews for?


In [17]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
pmt= "You are a helpful product assistant for an online tech store."
agent2 = create_agent(
    llm,
    tools = [get_product,get_review],
    system_prompt=pmt,
    checkpointer = checkpointer
)
def ask2(question : str,thread_id : str = "user-tiru-session-1"):
  config = {"configurable": {"thread_id":thread_id}}
  result = agent2.invoke({"messages" : [{"role" : "user",
                                        "content": question}]})
  print(result["messages"][-1].content)

In [18]:
# Turn 1: Establish context
ask2("I am looking at the wireless headphones.")

# Turn 2: Ask a follow-up without mentioning the product name again
ask2("What are the reviews on this product?")

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id